In [0]:
import requests
import time
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# 1. Coloca a tua chave da API do TMDB aqui
api_key = ""

print("🔍 Caçando os filmes com classificação 'null' na dimensão...")

# 2. Puxa só os filmes que estão sem idade na tabela DIM_FILME
df_nulos = spark.sql("""
    SELECT DISTINCT tmdb_id 
    FROM dbacademy.default.dim_filme 
    WHERE classificacao_etaria IS NULL
""")

lista_ids = [row.tmdb_id for row in df_nulos.collect() if row.tmdb_id is not None]
print(f"🎯 Total de filmes pra consultar na API: {len(lista_ids)}")

resultados = []

# 3. Bate na porta do TMDB pra cada filme
for tmdb_id in lista_ids:
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}/release_dates?api_key={api_key}"
    resp = requests.get(url)
    classificacao = None

    if resp.status_code == 200:
        data = resp.json()
        
        for country in data.get('results', []):
            if country.get('iso_3166_1') == 'BR':
                certifications = country.get('release_dates', [])
                if certifications:
                    for cert in certifications:
                        if cert.get('certification'):
                            classificacao = cert.get('certification')
                            break
                break 
    
    if classificacao:
        resultados.append((tmdb_id, classificacao))
    
    time.sleep(0.1)

print(f"✅ Conseguimos resgatar a idade de {len(resultados)} filmes!")

# 4. Atualiza a tabela DIM_FILME com os dados resgatados
if resultados:
    schema = StructType([
        StructField("tmdb_id", IntegerType(), True),
        StructField("nova_classificacao", StringType(), True)
    ])
    df_novos_dados = spark.createDataFrame(resultados, schema)
    
    df_novos_dados.createOrReplaceTempView("tmp_novas_classificacoes")
    
    print("🛠️ Fazendo o Update na tabela dim_filme...")
    
    # Fazendo o MERGE na tabela correta de dimensão!
    spark.sql("""
        MERGE INTO dbacademy.default.dim_filme target
        USING tmp_novas_classificacoes source
        ON target.tmdb_id = source.tmdb_id
        WHEN MATCHED THEN
          UPDATE SET target.classificacao_etaria = source.nova_classificacao
    """)
    
    print("🚀 Base atualizada! Agora sim, o painel vai brilhar!")
else:
    print("🤷‍♂️ A API do TMDB também não tinha as idades pra esses filmes.")

In [0]:
import requests

TMDB_API_KEY = ""
OMDB_KEY = ""
tmdb_id = 786892

print(f"🔍 Buscando dados no TMDB para o ID: {tmdb_id}...")

# 1. Pega o IMDb ID oficial direto do TMDB
url_tmdb = f"https://api.themoviedb.org/3/movie/{tmdb_id}?api_key={TMDB_API_KEY}&language=pt-BR"
resp_tmdb = requests.get(url_tmdb).json()

titulo_br = resp_tmdb.get('title')
titulo_original = resp_tmdb.get('original_title')
imdb_id = resp_tmdb.get('imdb_id')

print(f"🎬 Filme encontrado no TMDB: {titulo_br} ({titulo_original})")
print(f"🔑 IMDb ID correspondente: {imdb_id}")

# 2. Se achou o IMDb ID, consulta a OMDb de forma cirúrgica por ID (?i=)
if imdb_id:
    print(f"\n📡 Consultando a OMDb com o ID {imdb_id}...")
    url_omdb = f"http://www.omdbapi.com/?i={imdb_id}&apikey={OMDB_KEY}"
    resp_omdb = requests.get(url_omdb).json()
    
    if resp_omdb.get('Response') == 'True':
        print("\n✅ Dados retornados pela OMDb na régua:")
        print(f"• Título na OMDb: {resp_omdb.get('Title')}")
        print(f"• Ano: {resp_omdb.get('Year')}")
        print(f"• Nota IMDb: {resp_omdb.get('imdbRating')}")
        print(f"• Metascore: {resp_omdb.get('Metascore')}")
        print(f"• Outras Avaliações: {resp_omdb.get('Ratings')}")
    else:
        print(f"⚠️ A OMDb não achou registros para esse ID: {resp_omdb.get('Error')}")
else:
    print("⚠️ O TMDB não retornou um IMDb ID para esse filme.")

In [0]:
df_ancine_1gb = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ";") \
    .option("encoding", "UTF-8") \
    .load("/Volumes/dbacademy/default/puc/ancine_dados_brutos_2021_2026.csv")

# Recriando a tabela Bronze limpa com a acentuação correta
df_ancine_1gb.write.format("delta").mode("overwrite").saveAsTable("bronze_ancine")
print("✅ Base Bronze recriada com UTF-8: Adeus, texto quebrado!")

In [0]:
%sql
-- ========================================
-- CAMADA SILVER: Limpeza e Transformação
-- ========================================
-- Objetivo: Criar tabela silver_ancine com dados limpos, tipados e prontos para análise
-- Filtro Temporal: Dados já vêm filtrados (2021-2026) do arquivo fonte
-- Transformações: Remoção de duplicatas, tipagem de datas, seleção de colunas relevantes

CREATE OR REPLACE TABLE workspace.default.silver_ancine AS
SELECT DISTINCT
    -- Conversão de data do formato brasileiro (dd/MM/yyyy) para DATE
    TO_DATE(DATA_EXIBICAO, 'dd/MM/yyyy') AS DATA_EXIBICAO,
    
    -- Normalização de textos em UPPERCASE para padronização
    UPPER(TRIM(TITULO_ORIGINAL)) AS TITULO_ORIGINAL,
    UPPER(TRIM(TITULO_BRASIL)) AS TITULO_BRASIL,
    UPPER(TRIM(PAIS_OBRA)) AS PAIS_OBRA,
    UPPER(TRIM(RAZAO_SOCIAL_DISTRIBUIDORA)) AS RAZAO_SOCIAL_DISTRIBUIDORA,
    
    -- Métrica de público como INTEGER
    CAST(PUBLICO AS INT) AS PUBLICO
    
FROM bronze_ancine
WHERE 
    -- Filtros de qualidade de dados
    DATA_EXIBICAO IS NOT NULL
    AND PUBLICO IS NOT NULL
    AND PUBLICO > 0  -- Remove registros inválidos ou testes
    AND TITULO_ORIGINAL IS NOT NULL
ORDER BY DATA_EXIBICAO DESC;

-- Validação: Mostra estatísticas da camada Silver
SELECT 
    COUNT(*) AS total_registros,
    MIN(DATA_EXIBICAO) AS primeira_exibicao,
    MAX(DATA_EXIBICAO) AS ultima_exibicao,
    SUM(PUBLICO) AS publico_total,
    COUNT(DISTINCT TITULO_BRASIL) AS total_filmes_unicos
FROM workspace.default.silver_ancine;

In [0]:
import requests
import time
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# 1. Coloca a tua chave da API do TMDB aqui
api_key = "d586bba83710b6aa29d7c6776bcc3335"

print("🔍 Caçando os filmes com classificação 'null' na base...")

# 2. Puxa só os filmes que estão sem idade na tua tabela Gold (ou Silver, se preferir atualizar na origem)
df_nulos = spark.sql("""
    SELECT DISTINCT b.tmdb_id 
    FROM dbacademy.default.gold_features_bilheteria b
    JOIN dbacademy.default.dim_filme f ON b.tmdb_id = f.tmdb_id
    WHERE f.classificacao_etaria IS NULL
""")

lista_ids = [row.tmdb_id for row in df_nulos.collect() if row.tmdb_id is not None]
print(f"🎯 Total de filmes pra consultar na API: {len(lista_ids)}")

resultados = []

# 3. Bate na porta do TMDB pra cada filme
for tmdb_id in lista_ids:
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}/release_dates?api_key={api_key}"
    resp = requests.get(url)
    classificacao = None

    if resp.status_code == 200:
        data = resp.json()
        
        # Garimpa o JSON procurando o bloco do Brasil ("BR")
        for country in data.get('results', []):
            if country.get('iso_3166_1') == 'BR':
                certifications = country.get('release_dates', [])
                if certifications:
                    # Pega a primeira certificação que achar e que não seja vazia
                    for cert in certifications:
                        if cert.get('certification'):
                            classificacao = cert.get('certification')
                            break
                break # Achou o Brasil, não precisa olhar os outros países
    
    # Guarda o resultado se achou alguma coisa
    if classificacao:
        resultados.append((tmdb_id, classificacao))
    
    # Dá uma segurada pra API não te banir (100 milissegundos)
    time.sleep(0.1)

print(f"✅ Conseguimos resgatar a idade de {len(resultados)} filmes!")

# 4. Se achou dados novos, atualiza a base com um MERGE cabuloso
if resultados:
    # Cria um DataFrame do Spark com os dados resgatados
    schema = StructType([
        StructField("tmdb_id", IntegerType(), True),
        StructField("nova_classificacao", StringType(), True)
    ])
    df_novos_dados = spark.createDataFrame(resultados, schema)
    
    # Transforma num temp view pra usar no SQL
    df_novos_dados.createOrReplaceTempView("tmp_novas_classificacoes")
    
    print("🛠️ Fazendo o Update na tabela dim_filme...")
    
    # O Delta Lake faz a mágica de atualizar só onde der match no ID
    spark.sql("""
        MERGE INTO dbacademy.default.dim_filme target
        USING tmp_novas_classificacoes source
        ON target.tmdb_id = source.tmdb_id
        WHEN MATCHED THEN
          UPDATE SET target.classificacao_etaria = source.nova_classificacao
    """)
    
    print("🚀 Base atualizada! Pode rodar teu dashboard que o buraco diminuiu!")
else:
    print("🤷‍♂️ A API do TMDB também não tinha as idades pra esses filmes.")

In [0]:
%sql
SELECT 
    genero,
    SUM(renda) AS renda_total,
    SUM(publico) AS publico_total,
    ROUND(SUM(renda) / NULLIF(COUNT(id_sessao), 0), 2) AS renda_media_sessao,
    ROUND(SUM(publico) / NULLIF(COUNT(id_sessao), 0), 2) AS publico_medio_sessao
FROM tabela_cinema
WHERE genero IS NOT NULL
GROUP BY genero
ORDER BY renda_total DESC;